In [3]:
import torch
from models.model_dqn_lstm import DQNLSTM
from env_dqn import EVChargingEnv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'models.model_dqn_lstm'

In [ ]:
# Carga y limpieza del dataset
df = pd.read_csv("data_dqn/processed_ev_charging_patterns_dqn.csv")
df = df[df["Charging Rate (kW)"] <= 7.4].reset_index(drop=True)

In [ ]:
# Carga del modelo entrenado
model = DQNLSTM(input_dim=9, hidden_dim=64, lstm_layers=2, output_dim=2)
model.load_state_dict(torch.load("models/dqn_lstm_trained.pth"))
model.eval()

In [ ]:
#Simulación por episodios (SOC inicial real)
env = EVChargingEnv()
cost_policy, energy_policy, soc_policy = 0, 0, []
cost_real, energy_real, soc_real = 0, 0, []

for idx, row in df.iterrows():
    env.reset(soc=row["State of Charge (Start %)"])
    state = env._get_state().unsqueeze(0).unsqueeze(0)  # formato LSTM
    with torch.no_grad():
        q_values = model(state)
        action = torch.argmax(q_values).item()
    _, reward, _, _ = env.step(action)
    
    cost_policy += reward * -1
    energy_policy += env.power * 0.25
    soc_policy.append(env.soc)

    cost_real += row["Charging Cost (USD)"]
    energy_real += row["Energy Consumed (kWh)"]
    soc_real.append(row["State of Charge (End %)"])

In [ ]:
# Resultados finales
print("➡️ Política RL:")
print(f"   Coste total estimado: {cost_policy:.2f} USD")
print(f"   Energía suministrada: {energy_policy:.2f} kWh")

print("➡️ Datos reales:")
print(f"   Coste registrado: {cost_real:.2f} USD")
print(f"   Energía registrada: {energy_real:.2f} kWh")

In [ ]:
# Visualización de resultados
plt.figure(figsize=(18, 10))
plt.plot(soc_policy, label="SOC (RL)")
plt.plot(soc_real, label="SOC (Real)", linestyle="--")
plt.legend()
plt.title("Evolución del SOC")
plt.xlabel("Paso temporal")
plt.ylabel("SOC")
plt.grid()
plt.show()

fig, ax = plt.subplots()
ax.bar(["RL", "Real"], [cost_policy, cost_real])
ax.set_ylabel("Coste total (USD)")
ax.set_title("Comparativa de coste")
plt.grid()
plt.show()